In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [ ]:
# Load the Iris dataset
iris = load_iris()
data = iris.data  # Features
target = iris.target  # Labels


In [ ]:

# Simplify to binary classification (e.g., class 0 vs class 1)
binary_mask = target < 2
data_binary = data[binary_mask]
target_binary = target[binary_mask]
target_binary = np.where(target_binary == 0, -1, 1)  # Convert 0 -> -1 for SVM


In [ ]:

# Split into training and test sets
x_train, x_test, y_train, y_test = train_test_split(data_binary, target_binary, test_size=0.2, random_state=42)


In [ ]:

# Define helper functions
def hinge_loss(w, b, x, y):
    """Calculate the hinge loss for SVM."""
    margins = 1 - y * (np.dot(x, w) + b)
    return np.maximum(0, margins)

def gradient_descent_svm(x, y, lr=0.001, epochs=1000):
    """Train a basic linear SVM using gradient descent."""
    num_samples, num_features = x.shape
    w = np.zeros(num_features)  # Initialize weights
    b = 0  # Initialize bias

    for epoch in range(epochs):
        for i in range(num_samples):
            if y[i] * (np.dot(x[i], w) + b) < 1:
                # Misclassified
                w = w - lr * (2 * w - np.dot(x[i], y[i]))
                b = b + lr * y[i]
            else:
                # Correctly classified
                w = w - lr * 2 * w
    return w, b


In [ ]:

# Train the SVM
weights, bias = gradient_descent_svm(x_train, y_train)

# Predict
def predict(x, w, b):
    """Make predictions using the linear SVM."""
    return np.sign(np.dot(x, w) + b)

y_pred = predict(x_test, weights, bias)


In [ ]:
# Evaluate
accuracy = np.mean(y_pred == y_test)
print("Manual Linear SVM Accuracy:", accuracy * 100, "%")

In [ ]:
# Confusion Matrix
confusion = np.zeros((2, 2), dtype=int)
for true, pred in zip(y_test, y_pred):
    confusion[int((true + 1) / 2), int((pred + 1) / 2)] += 1
print("Confusion Matrix:\n", confusion)


In [ ]:

# Precision, Recall, and F1 Score
precision = confusion[1, 1] / (confusion[0, 1] + confusion[1, 1])
recall = confusion[1, 1] / (confusion[1, 0] + confusion[1, 1])
f1_score = 2 * precision * recall / (precision + recall)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)


In [ ]:


# Visualize decision boundary and data points
def plot_decision_boundary(x, y, w, b):
    plt.figure(figsize=(8, 6))

    # Plot data points
    for i in range(len(y)):
        if y[i] == 1:
            plt.scatter(x[i, 0], x[i, 1], color='blue', label='Class 1' if i == 0 else "")
        else:
            plt.scatter(x[i, 0], x[i, 1], color='red', label='Class -1' if i == 0 else "")

    # Create grid to evaluate model
    x_min, x_max = x[:, 0].min() - 1, x[:, 0].max() + 1
    y_min, y_max = x[:, 1].min() - 1, x[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # Calculate decision boundary
    zz = np.dot(grid, w) + b
    zz = zz.reshape(xx.shape)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1_score)

    # Plot decision boundary
    plt.contourf(xx, yy, zz > 0, alpha=0.2, levels=[-1, 0, 1], colors=['red', 'blue'])
    plt.contour(xx, yy, zz, levels=[-1, 0, 1], colors=['black'], linestyles=['--', '-', '--'])

    plt.title("Decision Boundary for Manual SVM")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.grid()
    plt.show()

# Plot decision boundary for the test data
plot_decision_boundary(x_test, y_test, weights[:2], bias)
